# PDB Manifest Generation (Phase 1)

Downloads the current wwPDB Beta holdings inventory, compares it against a
previous snapshot, and produces:

- `transfer_manifest.txt` — PDB IDs to download in Phase 2
- `removed_manifest.txt` — obsoleted PDB IDs to archive in Phase 3
- `updated_manifest.txt` — updated PDB IDs to pre-archive in Phase 3
- `diff_summary.json`    — human-readable summary of changes
- `holdings_snapshot.json.gz` — current snapshot for next run's diff

All filtering (hash range, limit) is applied here so downstream phases
receive a final, pre-filtered manifest.

## Typical workflow

1. Set `HASH_FROM`/`HASH_TO` to limit the batch (2-char hex range, e.g. `"00"` → `"3f"`)
2. Set `PREVIOUS_SNAPSHOT_URI` to the previous run's `holdings_snapshot.json.gz` in S3
3. Run all cells
4. Upload the output manifests and `holdings_snapshot.json.gz` to the staging location
5. Submit Phase 2 CTS job pointing at `transfer_manifest.txt`

## Path formats quick reference

| Suffix | Format | Example |
|--------|--------|---------|
| `_URI` | `s3://bucket/key/…` | `s3://cdm-lake/staging/run1/` |
| `_BUCKET` | bucket name only | `cdm-lake` |
| `_KEY_PREFIX` | S3 key prefix (no scheme/bucket) | `tenant-general-warehouse/kbase/datasets/pdb/` |
| `_DIR` / `_PATH` | local filesystem path | `output/transfer_manifest.txt` |

Lakehouse object: `s3://{STORE_BUCKET}/{STORE_KEY_PREFIX}raw_data/<hash>/<pdb_id>/…`

In [ ]:
"""Imports and S3 client initialisation."""

from __future__ import annotations

import json
from pathlib import Path

from cdm_data_loaders.pdb.manifest import (
    PDBDiffResult,
    build_current_records,
    compute_diff,
    download_holdings,
    download_holdings_snapshot,
    filter_by_hash_range,
    parse_removed_entries,
    scan_store_to_previous_ids,
    upload_holdings_snapshot,
    write_diff_summary,
    write_removed_manifest,
    write_transfer_manifest,
    write_updated_manifest,
)
from cdm_data_loaders.utils.s3 import get_s3_client, split_s3_path

In [ ]:
"""Configure parameters."""

# Hash-directory range (2-char hex, inclusive). Set to None to skip a bound.
# Full archive: hash_from=None, hash_to=None
HASH_FROM: str | None = None  # e.g. "00"
HASH_TO: str | None = None  # e.g. "3f"

# PDB file types to include in the transfer manifest.
# Set to None to download all types.
FILE_TYPES: list[str] | None = None  # e.g. ["structures", "validation_reports"]

# Maximum number of entries in the transfer manifest (None = unlimited)
LIMIT: int | None = None

# Previous holdings snapshot in S3 (from the last run).
# Set to None on the very first run.
# format: s3:// URI
PREVIOUS_SNAPSHOT_URI: str | None = None

# Scan the S3 Lakehouse for existing entries (bootstrap fallback).
# Only needed when PREVIOUS_SNAPSHOT_URI is None and there is existing data in S3.
SCAN_STORE = False

# Lakehouse S3 location (used for SCAN_STORE).
# format: bucket name (no s3:// scheme)
STORE_BUCKET: str | None = "cdm-lake"
# format: S3 key prefix within STORE_BUCKET (no scheme, no bucket)
STORE_KEY_PREFIX = "tenant-general-warehouse/kbase/datasets/pdb/"

# Local output directory for manifest files
# format: local directory path
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Hash range: {HASH_FROM!r} -> {HASH_TO!r}")
print(f"File types: {FILE_TYPES or 'all'}")
print(f"Limit: {LIMIT}")
print(f"Previous snapshot: {PREVIOUS_SNAPSHOT_URI or 'none (first run)'}")
print(f"Scan store: {SCAN_STORE}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
"""Validate S3 connectivity and bucket/prefix configuration."""

import boto3
from botocore.exceptions import ClientError, NoCredentialsError

s3 = boto3.client("s3")

try:
    sts = boto3.client("sts")
    identity = sts.get_caller_identity()
    print(f"✓ Credentials valid — account: {identity['Account']}, arn: {identity['Arn']}")
except NoCredentialsError:
    print("✗ No AWS credentials found")
    raise
except ClientError as e:
    if e.response["Error"]["Code"] == "InvalidParameterValue":
        print("✓ Credentials present (STS not supported on this endpoint — skipping identity check)")
    else:
        print(f"✗ Credential check failed: {e}")
        raise

_check_bucket = STORE_BUCKET or "cdm-lake"
_check_prefix = STORE_KEY_PREFIX
try:
    resp = s3.list_objects_v2(Bucket=_check_bucket, Prefix=_check_prefix, MaxKeys=1)
    if resp.get("KeyCount", 0) > 0:
        print(f"✓ Bucket accessible and prefix has objects: s3://{_check_bucket}/{_check_prefix}")
    else:
        print(f"✓ Bucket accessible but no objects at s3://{_check_bucket}/{_check_prefix} (expected on first run)")
except ClientError as e:
    code = e.response["Error"]["Code"]
    print(f"✗ S3 access check failed (HTTP {code}): {e}")
    raise

In [ ]:
"""Download current wwPDB holdings inventory."""

print("Downloading wwPDB Beta holdings files ...")
holdings_data = download_holdings()
current = build_current_records(holdings_data)
removed_ids = parse_removed_entries(holdings_data["removed"])

print(f"Current holdings: {len(current):,} entries")
print(f"Removed/obsoleted entries: {len(removed_ids):,}")

In [ ]:
"""Load previous holdings snapshot (or bootstrap from store scan)."""

previous: dict | None = None
previous_ids: set[str] | None = None

if PREVIOUS_SNAPSHOT_URI:
    s3_client = get_s3_client()
    bucket, key = split_s3_path(PREVIOUS_SNAPSHOT_URI)
    previous = download_holdings_snapshot(bucket, key)
    print(f"Loaded previous snapshot: {len(previous):,} entries from {PREVIOUS_SNAPSHOT_URI}")

elif SCAN_STORE and STORE_BUCKET:
    from tqdm.notebook import tqdm

    print(f"Scanning s3://{STORE_BUCKET}/{STORE_KEY_PREFIX} for existing PDB entries ...")
    progress = tqdm(unit="entry", desc="Scanning store", leave=True, mininterval=2.0)

    def _track_scan(count: int, pdb_id: str) -> None:
        progress.n = count
        progress.set_postfix(pdb_id=pdb_id, refresh=True)

    previous_ids = scan_store_to_previous_ids(
        STORE_BUCKET,
        STORE_KEY_PREFIX,
        progress_callback=_track_scan,
    )
    progress.n = len(previous_ids)
    progress.refresh()
    print(f"Found {len(previous_ids):,} existing PDB entries in the Lakehouse")

else:
    print("No previous snapshot — all current entries will be treated as new (first run)")

In [ ]:
"""Apply hash-range filter and compute diff."""

filtered = filter_by_hash_range(current, hash_from=HASH_FROM, hash_to=HASH_TO)
print(f"After hash filter: {len(filtered):,} entries")

filtered_prev = filter_by_hash_range(previous, hash_from=HASH_FROM, hash_to=HASH_TO) if previous else None
filtered_prev_ids = (
    {
        pid
        for pid in previous_ids
        if __import__("cdm_data_loaders.pdb.entry", fromlist=["pdb_id_hash"]).pdb_id_hash(pid) >= (HASH_FROM or "00")
        and __import__("cdm_data_loaders.pdb.entry", fromlist=["pdb_id_hash"]).pdb_id_hash(pid) <= (HASH_TO or "ff")
    }
    if previous_ids
    else None
)

diff = compute_diff(
    filtered,
    previous=filtered_prev,
    previous_ids=filtered_prev_ids,
    removed_ids=removed_ids,
)

print(f"New:     {len(diff.new):>8,}")
print(f"Updated: {len(diff.updated):>8,}")
print(f"Removed: {len(diff.removed):>8,}")
total = len(diff.new) + len(diff.updated)
print(f"--- Total to transfer: {total:,}")

if LIMIT:
    diff.new = diff.new[:LIMIT]
    diff.updated = diff.updated[: max(0, LIMIT - len(diff.new))]
    print(f"After LIMIT={LIMIT}: {len(diff.new) + len(diff.updated):,} entries in transfer manifest")

In [ ]:
"""Write manifest files to OUTPUT_DIR."""

transfer_manifest_path = OUTPUT_DIR / "transfer_manifest.txt"
removed_manifest_path = OUTPUT_DIR / "removed_manifest.txt"
updated_manifest_path = OUTPUT_DIR / "updated_manifest.txt"
diff_summary_path = OUTPUT_DIR / "diff_summary.json"

written = write_transfer_manifest(diff, transfer_manifest_path)
write_removed_manifest(diff, removed_manifest_path)
write_updated_manifest(diff, updated_manifest_path)
summary = write_diff_summary(diff, diff_summary_path, hash_from=HASH_FROM, hash_to=HASH_TO)

print(json.dumps(summary, indent=2))

In [ ]:
"""Save and optionally upload the current holdings snapshot for next run."""

from cdm_data_loaders.pdb.manifest import save_holdings_snapshot

snapshot_local = OUTPUT_DIR / "holdings_snapshot.json.gz"
save_holdings_snapshot(current, snapshot_local)
print(f"Saved holdings snapshot to {snapshot_local}")

# Optionally upload the snapshot to S3 for next run.
# Set SNAPSHOT_UPLOAD_URI to an s3:// URI to enable.
SNAPSHOT_UPLOAD_URI: str | None = None  # e.g. "s3://cdm-lake/staging/pdb-run1/holdings_snapshot.json.gz"

if SNAPSHOT_UPLOAD_URI:
    s3_client = get_s3_client()
    upload_bucket, upload_key = split_s3_path(SNAPSHOT_UPLOAD_URI)
    upload_holdings_snapshot(current, upload_bucket, upload_key)
    print(f"Uploaded holdings snapshot to {SNAPSHOT_UPLOAD_URI}")
else:
    print("SNAPSHOT_UPLOAD_URI not set — snapshot not uploaded to S3")

## Next steps

1. Upload the output files to the staging S3 prefix for Phase 2:
   ```
   aws s3 cp output/ s3://cdm-lake/staging/<run-id>/ --recursive
   ```
2. Submit the CTS Phase 2 job with:
   - **manifest**: `s3://cdm-lake/staging/<run-id>/transfer_manifest.txt`
   - **output_dir**: `s3://cdm-lake/staging/<run-id>/`
3. After Phase 2 completes, run `pdb_promote.ipynb` (Phase 3).